# Exercise 1: All-Atom Structural Prediction of a Peptide with Boltz-2

**Hardware:** GPU (T4 recommended) — set via *Runtime > Change runtime type*

---

### Our Test Case: Chignolin

We will fold **Chignolin** (`GYYDPETGTW`), a synthetic 10-residue mini-protein engineered to adopt a stable **beta-hairpin** conformation in solution. It is one of the smallest autonomously folding peptides known, making it an ideal benchmark for computational folding tools.

Its structure is stabilised by:
- A type-I' beta-turn centred on `DPE` (residues 4–6)
- Two backbone hydrogen bonds forming the hairpin
- A Tyr2–Trp10 aromatic stacking interaction

---

### Boltz-2: Architecture Overview

**Boltz-2** is a state-of-the-art biomolecular structure prediction model from MIT. Like AlphaFold3, it uses a **diffusion-based architecture** to generate all-atom coordinates. Key features:

| Feature | Description |
|---|---|
| Input | Sequence + optional MSA + optional templates |
| Architecture | Pairformer trunk + diffusion decoder |
| Output | All-atom coordinates + pLDDT + PAE |
| Multi-modal | Proteins, nucleic acids, small molecules, ions |

**pLDDT** (predicted Local Distance Difference Test) scores confidence per residue (0–100). **PAE** (Predicted Aligned Error) encodes inter-domain positional uncertainty as a 2D matrix.

---

## Step 1: Environment Setup

We use `micromamba` instead of `conda` because it resolves dependencies significantly faster in ephemeral Colab environments.

In [ ]:
# Install micromamba (fast conda alternative)
import os, subprocess

result = subprocess.run(
    'curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /usr/local/bin/ --strip-components=1 bin/micromamba',
    shell=True, capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "")
print(result.stderr[-500:] if result.stderr else "")

os.environ['MAMBA_ROOT_PREFIX'] = '/usr/local/envs'
print("micromamba ready")

In [ ]:
# Create boltz2 environment with all required dependencies
# This takes ~5-8 minutes on first run
cmd = """
micromamba create -n boltz2 -y \
  -c pytorch -c nvidia -c conda-forge -c pyg \
  python=3.10 \
  pytorch pytorch-cuda=12.1 \
  pyg \
  openmm \
  pdbfixer \
  2>&1 | tail -20
"""
result = subprocess.run(cmd, shell=True, capture_output=False, text=True)
print("Conda environment created")

In [ ]:
# Install boltz into the environment
result = subprocess.run(
    'micromamba run -n boltz2 pip install boltz 2>&1 | tail -10',
    shell=True, capture_output=False, text=True
)
print("boltz installed")

## Step 2: Prepare Input

In [ ]:
import os

os.makedirs('boltz_input', exist_ok=True)
os.makedirs('boltz_output', exist_ok=True)

# Write Chignolin FASTA
yaml_content = """
version: 1
sequences:
  - protein:
      id: [A]
      sequence: GYYDPETGTW

"""
with open('boltz_input/chignolin.yaml', 'w') as f:
    f.write(yaml_content)

print("Input YAML:")
print(yaml_content)
print(f"Sequence length: {len('GYYDPETGTW')} residues")

## Step 3: Run Boltz-2 Prediction


In [ ]:
# Run Boltz-2 prediction
cmd = """
micromamba run -n boltz2 boltz predict boltz_input/chignolin.yaml \
  --out_dir boltz_output \
  --use_msa_server \
  --num_workers 4
"""
result = subprocess.run(cmd, shell=True, capture_output=False)
print("\nPrediction complete. Checking output...")

# List output files
for root, dirs, files in os.walk('boltz_output'):
    for f in files:
        path = os.path.join(root, f)
        print(f"  {path}")

In [ ]:
!ls boltz_output/boltz_results_chignolin/predictions/

## Step 4: 3D Visualisation with py3Dmol

The predicted structure is colour-coded by **pLDDT** using the standard AlphaFold/Boltz convention:

| Colour | pLDDT | Interpretation |
|--------|-------|----------------|
| Blue | > 90 | Very high confidence |
| Cyan | 70–90 | Confident |
| Yellow | 50–70 | Low confidence |
| Orange | < 50 | Very low / disordered |

In [ ]:
!pip install py3dmol

In [ ]:
import py3Dmol
import glob

# Find the predicted CIF or PDB file
structure_files = glob.glob('boltz_output/boltz_results_chignolin/predictions/chignolin/*.cif', recursive=True)

if not structure_files:
    print("No structure file found. Check boltz_output/ directory.")
else:
    struct_path = structure_files[0]
    print(f"Visualising: {struct_path}")

    with open(struct_path, 'r') as f:
        struct_data = f.read()

    fmt = 'cif' if struct_path.endswith('.cif') else 'pdb'

    view = py3Dmol.view(width=700, height=500)
    view.addModel(struct_data, fmt)

    # Colour by B-factor (pLDDT stored in B-factor column)
    view.setStyle({'cartoon': {
        'colorscheme': {
            'prop': 'b',
            'gradient': 'roygb',
            'min': 50,
            'max': 100
        }
    }})
    view.addStyle({'stick': {'radius': 0.15, 'colorscheme': 'whiteCarbon'}})
    view.setBackgroundColor('0x1a1a2e')
    view.zoomTo()
    view.show()
    print("Rotate: left-drag | Zoom: scroll | Pan: right-drag")

## Step 5: Confidence Metric Analysis

### pLDDT — Per-residue Confidence

pLDDT is a per-residue score estimating how well the predicted local structure agrees with an experimental structure, if one were measured. It is **not** a thermodynamic stability score — it reflects model confidence in the coordinate placement.

### PAE — Predicted Aligned Error

PAE[i][j] gives the expected position error (in Å) at residue j when the structure is aligned on residue i. Low PAE off-diagonal blocks indicate **rigid domain–domain relationships**. For a 10-residue peptide, a uniformly low PAE indicates a single well-defined fold with no flexible linkers.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import glob

# Find pLDDT and PAE npz files
plddt_files = glob.glob('boltz_output/boltz_results_chignolin/predictions/chignolin/plddt_*.npz')
pae_files = glob.glob('boltz_output/boltz_results_chignolin/predictions/chignolin/pae_*.npz')

# Assuming confidence json is still needed for other metrics, although not for plddt/pae directly here.
conf_files = glob.glob('boltz_output/boltz_results_chignolin/predictions/chignolin/confidence_*.json')


if not plddt_files or not pae_files:
    print("pLDDT or PAE NPZ files not found. Generating synthetic example for illustration...")
    # Synthetic data matching expected Chignolin behaviour
    # Terminal residues (G1, W10) have lower pLDDT; core beta-turn is confident
    plddt = [72.1, 85.3, 88.7, 91.2, 90.5, 89.8, 87.4, 86.1, 83.9, 74.6]
    n = len(plddt)
    # PAE: low within hairpin, slightly higher at termini
    pae = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dist = abs(i - j)
            pae[i, j] = 1.5 + dist * 0.3 + np.random.normal(0, 0.2)
            if i in [0, 9] or j in [0, 9]:
                pae[i, j] += 2.0
    pae = np.clip(pae, 0, 15)
    residues = list(range(1, 11))
    sequence = 'GYYDPETGTW'
else:
    # Load pLDDT from npz
    plddt_path = plddt_files[0]
    print(f"Loading pLDDT from: {plddt_path}")
    with np.load(plddt_path) as data:
        # Assuming 'plddt' is the key used in boltz npz files
        plddt = data['plddt'].tolist()

    # Load PAE from npz
    pae_path = pae_files[0]
    print(f"Loading PAE from: {pae_path}")
    with np.load(pae_path) as data:
        # Assuming 'pae' is the key used in boltz npz files
        pae = data['pae']

    # The sequence 'GYYDPETGTW' is consistent throughout, so we can use it here.
    sequence = 'GYYDPETGTW'
    residues = list(range(1, len(sequence) + 1))

# ── Plotting ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22')
    for spine in ax.spines.values():
        spine.set_color('#30363d')

# ── Left: pLDDT line plot ──────────────────────────────────────────────────
ax1 = axes[0]
colors_per_residue = []
for v in plddt:
    if v >= 90: colors_per_residue.append('#4dabf7')
    elif v >= 70: colors_per_residue.append('#69db7c')
    elif v >= 50: colors_per_residue.append('#ffd43b')
    else: colors_per_residue.append('#ff6b6b')

ax1.fill_between(residues, plddt, alpha=0.15, color='#4dabf7')
ax1.plot(residues, plddt, color='#e6edf3', linewidth=2, zorder=3)
ax1.scatter(residues, plddt, c=colors_per_residue, s=80, zorder=4, edgecolors='white', linewidths=0.5)

# Confidence bands
for thresh, col, lbl in [(90, '#4dabf7', 'Very high'), (70, '#69db7c', 'Confident'),
                          (50, '#ffd43b', 'Low')]:
    ax1.axhline(thresh, color=col, linestyle='--', linewidth=0.8, alpha=0.5)

ax1.set_xticks(residues)
ax1.set_xticklabels([f"{r}\n{sequence[i]}" for i, r in enumerate(residues)],
                     color='#8b949e', fontsize=9)
ax1.set_ylim(0, 100)
ax1.set_xlabel('Residue', color='#8b949e')
ax1.set_ylabel('pLDDT Score', color='#8b949e')
ax1.set_title('Per-Residue pLDDT Confidence', color='#e6edf3', fontsize=12, pad=10)
ax1.tick_params(colors='#8b949e')
# Ensure residues are correctly defined before setting xlim
if residues:
    ax1.set_xlim(0.5, len(residues) + 0.5)
else:
    ax1.set_xlim(0, 1)

patches = [
    mpatches.Patch(color='#4dabf7', label='>90 Very high'),
    mpatches.Patch(color='#69db7c', label='70–90 Confident'),
    mpatches.Patch(color='#ffd43b', label='50–70 Low'),
]
ax1.legend(handles=patches, loc='lower center', facecolor='#161b22',
           labelcolor='#e6edf3', fontsize=8, framealpha=0.8)

# ── Right: PAE heatmap ────────────────────────────────────────────────────
ax2 = axes[1]
im = ax2.imshow(pae, cmap='Greens_r', vmin=0, vmax=15, aspect='auto')
ax2.set_xticks(range(len(residues)))
ax2.set_yticks(range(len(residues)))
ax2.set_xticklabels([f"{r}\n{sequence[i]}" for i, r in enumerate(residues)],
                     color='#8b949e', fontsize=8)
ax2.set_yticklabels([f"{r} {sequence[i]}" for i, r in enumerate(residues)],
                     color='#8b949e', fontsize=8)
ax2.set_title('Predicted Aligned Error (PAE) Matrix', color='#e6edf3', fontsize=12, pad=10)
ax2.set_xlabel('Scored residue (j)', color='#8b949e')
ax2.set_ylabel('Aligned residue (i)', color='#8b949e')

cbar = plt.colorbar(im, ax=ax2, shrink=0.85)
cbar.set_label('Expected position error (Å)', color='#8b949e')
cbar.ax.yaxis.set_tick_params(color='#8b949e')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#8b949e')

plt.suptitle('Boltz-2 Confidence Metrics — Chignolin (GYYDPETGTW)',
             color='#e6edf3', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('boltz_confidence_analysis.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

if plddt:
    print(f"\nMean pLDDT: {np.mean(plddt):.1f}")
    print(f"Min pLDDT: {np.min(plddt):.1f} (residue {np.argmin(plddt)+1})")
    print(f"Max pLDDT: {np.max(plddt):.1f} (residue {np.argmax(plddt)+1})")
else:
    print("\npLDDT data not available to calculate statistics.")